# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The FAIR² dataset contains clinical, pathological, and molecular biomarker variables for 77 cancer survivors with second primary colorectal cancer. We will load the Croissant schema, inspect record sets and fields by their `@id`s, extract data for analysis, and visualize key attributes.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records with `mlcroissant`. This process fetches the Croissant schema and dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Data Published: {metadata.datePublished}")

## 2. Data Overview
Review the record sets, fields, and their `@id`s, as defined in the Croissant schema. This helps you understand the tabular structure and what data is available for analysis.

**Note:** All record sets, fields, and columns are referenced by their `@id` for consistency and reproducibility.

In [ ]:
# List available record sets with their @id and titles
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | Name: {rs.get('name', rs['@id'])}")

# For demonstration, list the first record set's fields and their @id
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set '{first_record_set_id}':")
    fields = [f for f in dataset.fields(record_set=first_record_set_id)]
    for field in fields:
        print(f"@id: {field['@id']} | Name: {field.get('name', field['@id'])} | Data type: {field.get('dataType', '')}")

## 3. Data Extraction
Load all data from the main record set into a pandas DataFrame for further analysis. We reference every entity by its `@id` as specified in the overview.

The main tabular data record set (referenced by `@id`) is used below.

In [ ]:
# Extract all tabular data from each record set (by @id)
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    if not df.empty:
        dataframes[record_set_id] = df

# Preview data columns from each loaded DataFrame
for rs_id, df in dataframes.items():
    print(f"\nRecord set: {rs_id}")
    print("Fields:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Process and explore the main dataset:
- Filter records based on a numeric criterion (e.g., Age > 50)
- Normalize a numeric field
- Group by a categorical attribute such as MSI/MMR status or sex (as available by `@id`)

**Tip:** Use variable assignments for field `@id`s to ensure your code is robust.

In [ ]:
# Select the main record set DataFrame
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df_main = dataframes[main_record_set_id]
    print(f"Main record set @id: {main_record_set_id}")
else:
    raise ValueError("No record sets loaded.")

# Identify likely numeric and grouping fields by @id
sample_cols = df_main.columns.tolist()
print('Fields in main DataFrame:', sample_cols)

# Common field IDs (fill in appropriate @id as needed):
age_field_id = None
sex_field_id = None
msi_status_field_id = None

# Attempt to auto-detect field ids for demonstration based on names containing 'age', 'sex', 'msi', etc.
for col in sample_cols:
    lname = col.lower()
    if (age_field_id is None) and ("age" in lname):
        age_field_id = col
    if (sex_field_id is None) and ("sex" in lname):
        sex_field_id = col
    if (msi_status_field_id is None) and ("msi" in lname or "mmr" in lname):
        msi_status_field_id = col

print(f"Using age field: {age_field_id}")
print(f"Using sex field: {sex_field_id}")
print(f"Using MSI/MMR field: {msi_status_field_id}")

# Ensure the age field is numeric
if age_field_id is not None and age_field_id in df_main:
    df_main[age_field_id] = pd.to_numeric(df_main[age_field_id], errors='coerce')
    threshold = 50
    filtered_df = df_main[df_main[age_field_id] > threshold].copy()
    print(f"\nFiltered records with {age_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize age field
    normalized_col = f"{age_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"\nNormalized {age_field_id} for filtered records:")
    display(filtered_df[[age_field_id, normalized_col]].head())

    # Group by MSI status or sex if available
    group_field = msi_status_field_id if msi_status_field_id in filtered_df.columns else sex_field_id
    if group_field is not None:
        grouped = filtered_df.groupby(group_field)[age_field_id].mean()
        print(f"\nGrouped mean {age_field_id} by {group_field}:")
        display(grouped)
else:
    print("No numeric field (e.g., age) found for EDA.")

## 5. Visualization
Visualize the distribution of age and its relationship with MSI/MMR status or sex (as available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the age field after filtering
if age_field_id in filtered_df:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[age_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {age_field_id} (> {threshold})')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

    # Boxplot of age by MSI/MMR or Sex if available
    if group_field in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=age_field_id, data=filtered_df)
        plt.title(f'{age_field_id} by {group_field} (> {threshold})')
        plt.show()

## 6. Conclusion
We loaded a Croissant-documented clinical dataset using the `mlcroissant` library, explored its structure using entity `@id`s, and performed basic EDA. Age distributions and groupings by MSI/MMR or sex can offer insights into patient demographics and phenotype prevalence. For further analysis, you can use these patterns to drive downstream biomarker, outcome, or stratification studies.